# OpenHPC環境の削除

---

構築したOpenHPC環境を削除します。

## パラメータの指定

削除を行うのに必要となるパラメータを入力します。

### VCCアクセストークンの入力

VCノード, VCディスクを削除するためにVC Controller(VCC)のアクセストークンが必要となります。
次のセルを実行すると表示される入力枠にVCCのアクセストークンを入力してください。

> アクセストークン入力後に Enter キーを押すことで入力が完了します。

In [ ]:
from getpass import getpass
vcc_access_token = getpass()

入力されたアクセストークンが正しいことを、実際にVCCにアクセスして確認します。

In [ ]:
from common import logsetting
from vcpsdk.vcpsdk import VcpSDK

vcp = VcpSDK(vcc_access_token)

上のセルの実行結果がエラーとなり以下のようなメッセージが表示されている場合は、入力されたアクセストークンに誤りがあります。

```
config vc failed: http_status(403)
2021/XX/XX XX:XX:XX UTC: VCPAuthException: xxxxxxx:token lookup is failed: permission denied```

エラーになった場合はこの節のセルを全て `unfreeze` してから、もう一度アクセストークンの入力を行ってください。

### グループ名

OpenHPCのUnitGroup名を指定します。

VCノードを作成時に指定したUnitGroup名を確認するために `group_vars` ファイル名の一覧を表示します。

In [ ]:
!ls -1 group_vars/*.yml | sed -e 's/^group_vars\///' -e 's/\.yml//' | sort

UnitGroup名を指定してください。

In [ ]:
# (例)
# ugroup_name = 'OpenHPC'

ugroup_name = 

## 構築環境の削除

構築したOpenHPC環境を削除します。

### VCノードの削除

起動したVCノードを削除します。

現在のUnitGroupの一覧を確認します。

In [ ]:
vcp.df_ugroups()

現在のVCノードの状態を確認します。

In [ ]:
ug = vcp.get_ugroup(ugroup_name)
ug.df_nodes()

登録されている全Featureの計算ノード用VCユニットを削除します。

In [ ]:
import contextlib
from time import sleep
%run scripts/group.py
gvars = load_group_vars(ugroup_name)

for fname, fconfig in gvars["slurm_features"].items():
    vc_unit_name = fconfig.get("vc_unit", fname)
    unit = ug.get_unit(vc_unit_name)
    if unit is None:
        continue
    for addr in unit.find_ip_addresses():
        with contextlib.suppress(Exception):
            unit.delete_units(ip_address=addr)
            sleep(5)
    ug.delete_units(vc_unit_name, force=True)
    print(f"✓ {vc_unit_name}: 削除しました")

マスターノードと UnitGroup の削除を行います。

In [ ]:
ug.cleanup()

削除後の UnitGroupの一覧を確認します。

In [ ]:
vcp.df_ugroups()

### VCディスクの削除

NFS用のVCディスクを削除します。

> VCディスクを作成していない場合は、何もしません。

現在の状態を確認します。

In [ ]:
from IPython.display import display
ug_disk = vcp.get_ugroup(ugroup_name + '_disk')
if ug_disk:
    display(ug_disk.df_nodes())

VCディスクを削除します。

In [ ]:
if ug_disk:
    ug_disk.cleanup()

削除後のUnitGroupの一覧を確認します。

In [ ]:
vcp.df_ugroups()

## mdx VMの削除

VCノードとしてmdxを使用している場合は、マスターノードと計算ノードのVMを削除します。
mdx以外のクラウドを利用している場合はこの章をスキップしてください。

### mdx操作の準備

In [ ]:
from getpass import getpass
mdx_token = getpass("mdx API token")

IPv4接続を強制するように設定します。

In [ ]:
%run scripts/mdx_ops.py
use_ipv4_only()

mdxプラグインを読み込みます。

In [ ]:
from vcpsdk.plugins.mdx_ext import MdxResourceExt
mdx = MdxResourceExt(mdx_token)
mdx.set_current_project_by_name(gvars['mdx_project_name'])

### VM削除対象の特定

In [ ]:
# 全Featureの計算ノード + マスターノードのホスト名リストを作成
hosts_to_remove = []

for fname, fconfig in gvars["slurm_features"].items():
    hostname_template = fconfig["hostname_template"]
    max_nodes = len(fconfig["ip_addresses"])
    for i in range(max_nodes):
        hosts_to_remove.append(hostname_template.format(i + 1))

# マスターノードも削除対象に追加
hosts_to_remove.append(gvars['master_hostname'])

print(f"削除対象VM ({len(hosts_to_remove)}台):")
for vm in hosts_to_remove:
    print(f"  - {vm}")

### VMの強制停止と削除

In [ ]:
from vcpsdk.plugins.mdx_ext import SLEEP_COUNT
from vcpsdk.plugins.mdx_ext import SLEEP_TIME_SEC
from time import sleep

# VM強制停止
for vm in hosts_to_remove:
    try:
        mdx.power_off_vm(vm, wait_for=False)
    except Exception as e:
        print(f"⚠ {vm} の停止でエラー: {e}")

for vm in hosts_to_remove:
    for i in range(SLEEP_COUNT):
        info = mdx.get_vm_info(vm)
        if info is None or info['status'] == 'PowerOFF':
            break
        sleep(SLEEP_TIME_SEC)
    else:
        raise RuntimeError(f'Error: VM {vm} not powered off.')

In [ ]:
# VM削除
for vm in hosts_to_remove:
    try:
        mdx.destroy_vm(vm, wait_for=False)
    except Exception as e:
        print(f"⚠ {vm} の削除でエラー: {e}")

for vm in hosts_to_remove:
    for i in range(SLEEP_COUNT):
        info = mdx.get_vm_info(vm)
        if info is None:
            print(f"✓ {vm} を削除しました")
            break
        sleep(SLEEP_TIME_SEC)
    else:
        raise RuntimeError(f'Error: VM {vm} not destroyed.')

## Ansible設定のクリア

削除した環境に対応するAnsibleの設定をクリアします。

### group_varsファイル

group_varsファイルをリネームします。

In [ ]:
!mv group_vars/{ugroup_name}.yml group_vars/{ugroup_name}.yml.bak

### インベントリ

インベントリから UnitGroup に対応するグループを削除します。

In [ ]:
%run scripts/group.py
remove_group_from_inventory_yml(ugroup_name)

!cat inventory.yml